# IMO Health — DS LangChain Agent

A simple general-purpose agent powered by:
- **AWS Bedrock** (Claude) as the LLM
- **IMO Health MCP Gateway** for clinical terminology tools
- **LangGraph ReAct agent** for orchestration

Ask it anything — it will call MCP tools when relevant.

# IMO Health — Diagnosis Specificity Agent

This notebook provides a **Diagnosis Specificity Agent** that finds the most specific ICD-10 code for clinical diagnoses by traversing the IMO Knowledge Graph.

## What Does This Agent Do?

Given a clinical note, this agent will:
1. **Extract**-base diagnoses from the note (stripping qualifiers like laterality, severity, etc.)
2. **Let you pick**-which diagnosis to refine
3. **Normalize**-the diagnosis using IMO's Precision Normalize API
4. **Query refinements**-from the IMO Knowledge Graph
5. **Match refinements to evidence**-in the clinical note
6. **Resolve**-the most specific code via graph traversal
7. **Present**-the final recommendation with ICD-10 codes and evidence




| Component | Technology |
|-----------|------------|
| LLM | AWS Bedrock (Claude Haiku 4.5) |
| Tools | IMO Health MCP Gateway (4 tools) |
| Agent | LangGraph ReAct Agent |
| Auth | OAuth2 client_credentials grant |

## Prerequisites

- `config.json` file with MCP credentials 
- AWS credentials (SageMaker execution role or config.json)
- Python 3.10+


## Step 1: Install Dependencies

Run this cell once, then restart the kernel.

In [1]:
%pip install -q --upgrade --index-url https://pypi.org/simple/ \
    "langchain-core>=1.0.0" \
    "langchain>=1.0.0" \
    "langchain-aws>=0.2.0" \
    "langchain-mcp-adapters>=0.1.5" \
    "langgraph>=0.2.0" \
    "mcp<2.0.0" \
    boto3 botocore requests nest_asyncio httpx ipywidgets


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Configuration

Loads credentials from `config.json`. Create one from `config.json.template` if it doesn't exist.

**On Local:** Falls back to `config.json` file.

In [2]:
import os
import re
import json
import glob
import uuid
import time
import asyncio
import pathlib
import nest_asyncio
from collections import Counter
from datetime import datetime

import requests

nest_asyncio.apply()

# --- Load config.json ---
candidates = [
    pathlib.Path(__file__).parent / 'config.json' if '__file__' in dir() else pathlib.Path('config.json'),
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json',
    pathlib.Path(r'd:\Users\nthonte\IMO-Work\Solution-Engineering\solution-accelerators\Diagnosis Specificity Agent\notebook\config.json'),
]
cfg_path = next((p for p in candidates if p.exists()), None)
if cfg_path is None:
    raise FileNotFoundError('config.json not found. Copy config.json.template to config.json and fill in your credentials.')

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

mcp_cfg = cfg.get('mcp', {})

MCP_CLIENT_ID     = mcp_cfg.get('client_id', '')
MCP_CLIENT_SECRET = mcp_cfg.get('client_secret', '')
TOKEN_URL         = mcp_cfg.get('token_url', 'https://api.imohealth.com/oauth/token')
MCP_SERVER_URL    = mcp_cfg.get('server_url', '')
BEDROCK_MODEL     = mcp_cfg.get('bedrock_model_id', '')
BEDROCK_REGION    = mcp_cfg.get('aws_region', 'us-east-1')

if not MCP_CLIENT_SECRET:
    raise ValueError('client_secret is missing in config.json under the "mcp" section.')

# --- Load AWS credentials from config.json (for local dev) ---
aws_cfg = cfg.get('aws', {})
if aws_cfg.get('access_key_id'):
    os.environ.pop('AWS_PROFILE', None)
    os.environ['AWS_ACCESS_KEY_ID'] = aws_cfg['access_key_id']
    os.environ['AWS_SECRET_ACCESS_KEY'] = aws_cfg['secret_access_key']
    os.environ['AWS_SESSION_TOKEN'] = aws_cfg.get('session_token', '')
    os.environ['AWS_DEFAULT_REGION'] = aws_cfg.get('region', 'us-east-1')
    print(f'AWS credentials   : loaded from config.json (key prefix: {aws_cfg["access_key_id"][:8]}...)')
else:
    print('AWS credentials   : using default chain (env/IAM role)')

print(f'Config loaded from : {cfg_path.resolve()}')
print(f'Bedrock Model      : {BEDROCK_MODEL}')
print(f'AWS Region         : {BEDROCK_REGION}')
print(f'MCP Server URL     : {MCP_SERVER_URL}')


AWS credentials   : loaded from config.json (key prefix: ASIASXN2...)
Config loaded from : D:\Users\nthonte\IMO-Work\Solution-Engineering\solution-accelerators\Diagnosis Specificity Agent\notebook\config.json
Bedrock Model      : arn:aws:bedrock:us-east-1:187759307732:application-inference-profile/cy1tb7yulq51
AWS Region         : us-east-1
MCP Server URL     : https://api.imohealth.com/mcp


## Step 3: Get OAuth Token

Authenticates with the IMO Health API using the OAuth2 `client_credentials` grant.  
The token is used to authorize MCP Gateway requests.

In [3]:
def get_token(client_id: str, client_secret: str) -> str:
    payload = {
        'grant_type':    'client_credentials',
        'client_id':     client_id,
        'client_secret': client_secret,
        'audience':      'https://api.imohealth.com'
    }
    resp = requests.post(TOKEN_URL, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()['access_token']

access_token = get_token(MCP_CLIENT_ID, MCP_CLIENT_SECRET)
print('Token acquired (prefix):', access_token[:20] + '...')

Token acquired (prefix): eyJhbGciOiJSUzI1NiIs...


## Step 4: Connect to MCP Gateway & Discover Tools

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_CLIENT_CONFIG = {
    'mcp-gateway': {
        'url':       MCP_SERVER_URL,
        'transport': 'streamable_http',
        'headers':   {'Authorization': f'Bearer {access_token}'},
        'timeout':   300,
    }
}

async def get_langchain_tools():
    client = MultiServerMCPClient(MCP_CLIENT_CONFIG)
    tools = await client.get_tools()
    print(f'Discovered {len(tools)} MCP tools:\n')
    for t in tools:
        print(f'  - {t.name}')
    return tools

mcp_tools_list = asyncio.run(get_langchain_tools())


Session termination failed: 502


Discovered 15 MCP tools:

  - categorize___categorize_problems
  - ccp___entity_extraction
  - core-search___get_term_detail
  - core-search___lookup_term_by_code
  - core-search___search_medical_term
  - graphql-modifier___get_allowed_refinements
  - graphql-modifier___get_cross_domain
  - graphql-modifier___get_lexical
  - graphql-modifier___get_mappings
  - graphql-modifier___get_narrower_hierarchy
  - graphql-modifier___get_narrower_sequential_refinements
  - graphql-modifier___get_narrower_with_refinements
  - graphql-modifier___get_refinement_group
  - graphql-modifier___get_related_problems
  - normalize-ppml___normalize_ppml_term


## Step 5: Connect to MCP Gateway & Load DS Tools

Connects to the IMO Health MCP Gateway and loads **tools** needed for diagnosis specificity:

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_CLIENT_CONFIG = {
    'mcp-gateway': {
        'url': MCP_SERVER_URL,
        'transport': 'streamable_http',
        'headers': {'Authorization': f'Bearer {access_token}'},
        'timeout': 300,
    }
}

DS_TOOL_NAMES = [
    'normalize-ppml___normalize_ppml_term',
    'graphql-modifier___get_lexical',
    'graphql-modifier___get_allowed_refinements',
    'graphql-modifier___get_narrower_sequential_refinements',
]

async def get_ds_tools():
    client = MultiServerMCPClient(MCP_CLIENT_CONFIG)
    all_tools = await client.get_tools()
    ds_tools = [t for t in all_tools if t.name in DS_TOOL_NAMES]
    print(f'Loaded {len(ds_tools)}/{len(all_tools)} DS-specific tools:')
    for t in ds_tools:
        print(f'  - {t.name}')
    return ds_tools

mcp_tools = asyncio.run(get_ds_tools())

Session termination failed: 502


Loaded 4/15 DS-specific tools:
  - graphql-modifier___get_allowed_refinements
  - graphql-modifier___get_lexical
  - graphql-modifier___get_narrower_sequential_refinements
  - normalize-ppml___normalize_ppml_term


## Step 6: Initialize LLM (AWS Bedrock)

Creates the LLM client using AWS Bedrock with Claude Haiku 4.5.  
Temperature is set to 0 for deterministic, consistent outputs.

In [6]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model_id=BEDROCK_MODEL,
    region_name=BEDROCK_REGION,
    temperature=0,
    provider="anthropic",
)

print(f'LLM ready: {BEDROCK_MODEL}')


LLM ready: arn:aws:bedrock:us-east-1:187759307732:application-inference-profile/cy1tb7yulq51


## Step 7: Create the DS Agent

This cell creates the Diagnosis Specificity Agent with the full production system prompt.

### Agent Workflow (4 Phases)

Phase 1: EXTRACT   → Parse note, list base diagnoses, ask user to pick

Phase 2: NORMALIZE → normalize-ppml → get default_lexical_code

Phase 3: REFINE    → get_lexical → get refinement groups from KG

Phase 4: RESOLVE   → Match evidence → get_narrower_sequential → verify ICD-10

In [7]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a clinical diagnostic specificity agent that uses the IMO Normalize API and Knowledge Graph to find the most specific diagnosis for clinical encounters.

Your greeting (already shown to the user):
"I am a clinical diagnostic specificity agent that can find the most specific diagnosis for a patient visit. Provide the patient visit summary or clinical note and I will use IMO Normalize API and Knowledge Graph to get the highest level of diagnostic specificity."

## CRITICAL RULES — READ THESE FIRST
1. **Extraction Rule:** When a user sends a clinical note or encounter summary, you MUST respond with ONLY a text message listing extracted base problems. You are STRICTLY FORBIDDEN from calling any tools (normalize-ppml___normalize_ppml_term, graphql-modifier___get_lexical, graphql-modifier___get_narrower_sequential_refinements) until the user replies and tells you which finding to refine. The ONLY exception: the user explicitly names a single diagnosis to refine (e.g., "refine hypertension"). A clinical note that contains an assessment is NOT a refinement request — it still requires extraction first.

2. **Tool Explanation Rule:** BEFORE calling ANY tool, you MUST output a brief explanation (1-2 sentences) of WHY you are calling that tool. This helps users understand your reasoning process. Never call a tool without first explaining what you're about to do and why.

3. **CRITICAL: Always Use default_lexical_code:** When calling graphql-modifier___get_lexical, you MUST use the `default_lexical_code` field from the normalize results, NOT the `lexical_code` field. The normalize API returns both fields, but `default_lexical_code` is the canonical code that must be used for Knowledge Graph lookups.

## Your Workflow

### Phase 1: Extract the Base Problem (TEXT ONLY — NO TOOL CALLS)
When the user provides a clinical note or encounter summary:

**Case A — User explicitly names a diagnosis to refine** (e.g., "find most specific diagnosis for diabetes", "refine hypertension"):
- The user has already told you what to refine. Do NOT list findings or ask.
- **Skip directly to Phase 2** using the diagnosis the user mentioned.
- Proceed through Phase 2 → 3 → 4 in one continuous flow.

**Case B — User provides a clinical note (THIS IS THE DEFAULT CASE):**
- Read the **entire** note carefully — look at History, HPI, Assessment, and Plan sections
- Identify all **diagnoses and conditions** mentioned anywhere in the note (not just the Assessment). Do NOT extract individual symptoms, signs, vitals, or lab values (e.g., do NOT extract "productive cough", "fever", "crackles", "elevated WBC" as separate findings).
- Extract ONLY the **base diagnoses/conditions** — strip away ALL qualifiers such as severity (e.g., "sharp", "acute"), laterality (e.g., "left-sided", "right", "bilateral"), anatomical location details (e.g., "right lower lobe"), chronicity ("chronic", "recurrent"), typing (e.g., "type 2"), and associated conditions.
- Examples of correct extraction:
  - Note: "Patient presents with sharp, left-sided chest pain... Assessment: Chest pain, likely musculoskeletal vs anginal" → extract **"Chest pain"**
  - Note: "...Assessment: Type 2 diabetes with diabetic peripheral neuropathy" → extract **"Diabetes mellitus"** (NOT "Type 2 diabetes", NOT "tingling in feet")
  - Note: "65-year-old female with essential hypertension and chronic kidney disease stage 3... Assessment: Hypertensive chronic kidney disease with stage 3 CKD" → extract **"Hypertension"** and **"Chronic kidney disease"** (NOT "hypertensive chronic kidney disease")
  - Note: "...Assessment: Community-acquired pneumonia, right lower lobe" → extract **"Pneumonia"** (NOT "community-acquired pneumonia, right lower lobe", NOT "productive cough", NOT "fever")
  - Note: "Nasal congestion from fall pollen and allergies" → extract **"Nasal congestion"** (NOT "Nasal congestion (from fall pollen and allergies)")
- **CRITICAL OUTPUT FORMAT:** Present a numbered summary of the extracted BASE diagnoses to the user. List ONLY the base diagnosis names with NO additional text, NO qualifiers, NO explanations, and NO parenthetical information whatsoever.
  - CORRECT: "1. Nasal congestion"
  - WRONG: "1. Nasal congestion (from fall pollen and allergies)"
  - WRONG: "1. Congestive heart failure (CHF)"
  - WRONG: "1. Depression (patient reports feeling down)"
- Ask: "Please specify which finding(s) you would like to refine."
- **YOU MUST STOP HERE. DO NOT CALL ANY TOOLS. WAIT FOR THE USER TO REPLY.**

### Phase 2: Normalize & Identify (IMO Normalize API)
1. **BEFORE calling the normalize tool**, output a brief explanation (1-2 sentences) of why you need to call it
   - Example: "I will normalize the term 'Hypertension' using the IMO Normalize API to get the lexical code for Knowledge Graph lookup."
   - Example: "I will normalize the term 'Chest pain' to retrieve the standardized lexical code needed to query refinements."
2. Use **normalize-ppml___normalize_ppml_term** to normalize the base problem via the IMO Precision Normalize API
   - Use domain "Problem" for diagnoses/conditions
   - Pass the term as a single-element list in the `terms` parameter (e.g., `terms: ["chest pain"]`)
   - Use the `domain` parameter with value "Problem", "Procedure", "Medication", or "Lab"
   - Normalize the BASE term only (e.g., "chest pain", "diabetes mellitus", "hypertension", "pneumonia")
3. From the normalize results, identify the best matching base result (highest score, most relevant title and ICD-10 codes)

### Phase 3: Query Refinements via Knowledge Graph
3. **BEFORE calling the graphql tool**, output a brief explanation (1-2 sentences) of why you need to call it
   - Example: "Now retrieving refinements from the Knowledge Graph to identify available hypertension types and subtypes."
   - Example: "Now querying the Knowledge Graph using lexical code 29688 to retrieve all available refinement groups for diabetes mellitus."
4. Use **graphql-modifier___get_lexical** with the **default_lexical_code** from the best normalize result to retrieve refinements
   - **CRITICAL:** You MUST use the `default_lexical_code` field from the normalize response, NOT `lexical_code`
   - The normalize API returns BOTH fields: `lexical_code` and `default_lexical_code`
   - Example from normalize response: `{"title": "Hypertension", "lexical_code": "86491", "default_lexical_code": "86491", ...}`
   - **Always extract and use the `default_lexical_code` value** (e.g., `"default_lexical_code": "86491"`)
   - Pass the `default_lexical_code` value as the `imo_lexical_code` parameter to graphql-modifier___get_lexical
5. The allowed refinements returned by graphql-modifier___get_lexical are already grouped by category — use them directly to analyze which refinements match the clinical note
6. Do NOT simply dump the raw refinement data to the user — you must analyze it internally

### Phase 4: Reason & Recommend (THIS IS THE MOST IMPORTANT PHASE)
After gathering all refinement data, you MUST do the following:

**Step 1 — Match refinements to clinical evidence:**
For EACH refinement group, review the original clinical note and determine:
- Does the note contain words, phrases, or clinical findings that support a specific refinement? If yes, select it and cite the supporting evidence.
- If no refinement in the group has supporting evidence in the note, mark it as an "evidence gap". Do NOT select any refinement for that group.

**Refinement selection rules:**
- Select a refinement when the note provides supporting clinical evidence — this can be a direct quote OR a clear clinical implication from documented findings.
- Examples of valid clinical implications:
  - "sharp chest pain worse with exertion" → supports **"Acute"** (onset pattern implies acute presentation)
  - "left-sided chest pain" → supports **"Left"** laterality
  - "chronic kidney disease stage 3, eGFR 45" in a hypertension patient → supports **"secondary to renal failure"** (CKD is the underlying renal condition)
  - "right lower lobe infiltrate" → supports **"Right"** laterality and **"Lower lobe of lung"** location
  - "tingling in both feet, decreased sensation" in a diabetes patient → supports **"with peripheral neuropathy"**
- NEVER select "unspecified", "other", "NOS", or catch-all/default refinements. If the note doesn't provide enough info to pick a specific refinement, it's an evidence gap.
- NEVER fabricate clinical details not supported by the note. The note must contain SOME evidence (direct or implied) for the refinement.
- Do NOT add refinements that contradict or go beyond what the note describes.

**Step 2 — Build the most specific diagnosis(es):**
Combine the base diagnosis with selected refinements to form the most specific diagnosis possible for each valid path. The base diagnosis MUST always be included in each final combined name. For example, if the base is "Chest pain" and the selected refinement is "due to unstable angina pectoris", the most specific diagnosis is "Chest pain due to unstable angina pectoris" — NOT just "Unstable angina pectoris". The refinement refines the base; it does not replace it.

**Step 2.5 — Resolve the FINAL concept using graph traversal (MANDATORY):**
- You MUST use **graphql-modifier___get_narrower_sequential_refinements** to resolve the final concept.
- This tool performs NESTED/CHAINED traversal: it applies refinements sequentially, where each refinement narrows down from the results of the previous refinement.
- CRITICAL: Make ONE call with ALL selected refinements in a single sequence. Do NOT make separate calls for each refinement.
- Tool parameters:
  - `imo_lexical_code` (string): The base concept code to start from
  - `refinement_sequence` (array of arrays): ALL selected refinements in order, where each inner array contains one refinement code per step.
  - `include_mappings` (boolean): Set to true to include ICD-10/other code mappings in the response
- Example with 4 refinements:
  ```
  {
    "imo_lexical_code": "601056",
    "refinement_sequence": [["230"], ["362636588"], ["1328819463"], ["refinement4"]],
    "include_mappings": true
  }
  ```
  This applies: base concept → refinement 230 → refinement 362636588 → refinement 1328819463 → refinement4 (chained)
- The agent should determine the optimal order for applying refinements based on clinical logic and refinement group hierarchy (e.g., type → complication → laterality → severity).
- If you have conflicting options within the same refinement group (e.g., "right" vs "left" laterality), you may make separate calls to explore different paths. But all non-conflicting refinements MUST be included in the same sequence.
- Never skip this step. Never pick the final concept by text-only composition when this tool can be used.

**Step 3 — Verify the refined diagnosis codes:**
BEFORE calling the normalize tool for verification, output a brief explanation:
- Example: "Analyzing refinements against the clinical note. Now verifying the refined diagnosis code:"
- Example: "Now verifying the final refined diagnosis to retrieve the standardized ICD-10 code:"
Then use **normalize-ppml___normalize_ppml_term** to normalize each final graph-resolved diagnosis from Step 2.5 (pass as `terms: ["<diagnosis>"]`, `domain: "Problem"`).
This verification normalization retrieves the standardized title and ICD-10 code for the refined diagnosis. Do NOT reuse the base diagnosis codes — they are for the unmodified term only. Do NOT invent or guess codes.

**Step 4 — Present the final recommendation in this EXACT format:**

#### Refinement Analysis
| Refinement Group | Selected Refinement | Evidence from Note | Reasoning |
|---|---|---|---|
| (group name) | (chosen refinement or "No evidence") | (quote from note) | (why selected/rejected) |

#### Final Recommendation
- **Base Diagnosis:** (name) — ICD-10: (code from normalize results)
- **Selected Refinements:** (list each refinement selected with evidence)
- **Most Specific Diagnosis:** (the full refined diagnosis — MUST include the base diagnosis name combined with all selected refinements, e.g., "Chest pain due to unstable angina pectoris", NOT just the refinement alone) — ICD-10: (code from verification normalization)
- **Evidence Gaps (informational only):** The Knowledge Graph has refinements for these categories, but the clinical note does not contain evidence to support them. These do NOT mean the diagnosis is incomplete.

If one or more tested combinations resolve to concepts, provide:
- **Most Specific Diagnoses (Resolved Combination Paths):** list EACH resolved path separately, including the exact selected refinements for that path, resolved concept title/code, and verified ICD-10 code.

For each resolved path, you MUST use this path format:
- `Path N — <focus label>:`
- `Refined Diagnosis: <resolved diagnosis title>`
- `IMO Lexical Code: <lexical code>`
- `IMO KG Link: https://studio.imohealth.com/#/terminology-browser/graph?id=<lexical code>`
- `ICD-10 Codes:`
    - `Primary: <first ICD-10 code>`
    - `Secondary: <second ICD-10 code>` (if present)
    - `Tertiary: <third ICD-10 code>` (if present)
    - `Additional: <remaining ICD-10 codes comma-separated>` (if more than 3)
- `Evidence: <concise supporting evidence from note>`

ICD-10 ordering and highlighting rules:
- When multiple ICD-10 codes exist for a term, always label the first as **Primary**, second as **Secondary**, and third as **Tertiary**.
- Do not collapse multi-code outputs into a single unlabeled ICD line.
- If only one ICD-10 code exists, show Primary only.

## Important Rules
- NEVER call any tools when the user first provides a clinical note — ALWAYS extract base problems and ask first
- In Phase 1, ALWAYS extract only the base problem — never include severity, laterality, anatomical details, or typing qualifiers in the extracted finding
- Specificity comes from REFINEMENTS discovered via the Knowledge Graph, not from the initial normalization
- ONLY select a refinement when there is supporting evidence (direct quote or clear clinical implication) from the note — never select "unspecified", "other", or "NOS" refinements
- Do NOT just list raw refinement groups to the user — always analyze them against the note
- The goal is to return the most specific diagnosis set supported by evidence and graph resolution: include every valid tested combination path that resolves to a concept.
- If the note lacks information for a refinement group, leave it out of the diagnosis entirely and list it as an "evidence gap"
- Do NOT guess or hallucinate clinical information not present in the note
- NEVER fabricate ICD-10 codes — always get them from normalize tool results
- CRITICAL: You MUST ALWAYS call the tools (normalize-ppml___normalize_ppml_term, graphql-modifier___get_lexical, and graphql-modifier___get_narrower_sequential_refinements) for EVERY refinement request, even if you have seen similar data in previous conversation turns. NEVER skip tool calls or reuse results from memory — always make fresh API calls. Each refinement MUST follow Phase 2 → 3 → 4 with actual tool calls.
- If a tool returns too much data or errors due to payload size, summarize the top-level categories available and ask the user to pick a specific pathway to drill into.
"""

agent = create_react_agent(llm, mcp_tools, prompt=SYSTEM_PROMPT)
print('DS Agent ready.')

DS Agent ready.


D:\Users\nthonte\AppData\Local\Temp\1\ipykernel_17760\3278037045.py:165: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, mcp_tools, prompt=SYSTEM_PROMPT)


## Step 8: Interactive Chat

Run this cell to start the interactive agent.

### Output Legend
- 🔧 **Blue boxes** — Tool calls (MCP tool name + parameters)
- ✅ **Green boxes** — Tool results (click "View full result" to expand)
- 📝 **Formatted text** — Agent's final recommendation with tables


In [8]:
# Cell 8 — Rich UI Chat (no ipywidgets required)

from IPython.display import display, HTML, Markdown
import html as html_module
import json


class AgentUI:
    """Rich HTML display for agent streaming output."""

    @staticmethod
    def header():
        display(HTML("""
        <div style="text-align:center; padding:20px; background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    border-radius:12px; color:white; margin-bottom:16px;">
            <h2 style="margin:0;">Diagnosis Specificity Agent</h2>
            <p style="margin:4px 0 0 0; opacity:0.9;">Paste a clinical note → Select diagnosis → Get most specific code</p>
            <p style="margin:4px 0 0 0; opacity:0.7; font-size:12px;">Type <b>quit</b> to end · <b>reset</b> to clear history · <b>Kernel Interrupt</b> to stop mid-generation</p>
        </div>
        """))

    @staticmethod
    def status(message):
        display(HTML(f'<div style="color:#5f6368; font-size:12px; font-style:italic; padding:4px 0;">{html_module.escape(message)}</div>'))

    @staticmethod
    def tool_call(tool_name, parameters):
        params_json = html_module.escape(json.dumps(parameters, indent=2))
        display(HTML(f"""
        <div style="background:#e8f0fe; border:1px solid #1a73e8; border-radius:8px; padding:12px 16px; margin:8px 0; font-family:monospace; font-size:13px;">
            <b style="color:#1a73e8;">🔧 Tool Call: {html_module.escape(tool_name)}</b>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:12px; overflow-x:auto;">{params_json}</pre>
        </div>
        """))

    @staticmethod
    def tool_result(tool_name, result):
        result_str = str(result)
        preview = html_module.escape(result_str[:300])
        full = html_module.escape(result_str)
        char_count = len(result_str)
        display(HTML(f"""
        <div style="background:#e6f4ea; border:1px solid #137333; border-radius:8px; padding:12px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#137333;">✅ Result: {html_module.escape(tool_name)}</b>
            <details style="margin-top:8px;">
                <summary style="cursor:pointer; font-weight:600; font-size:12px; color:#137333;">View full result ({char_count:,} chars)</summary>
                <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:11px; max-height:300px; overflow:auto;">{full}</pre>
            </details>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:4px; font-size:11px; opacity:0.7;">{preview}{'...' if char_count > 300 else ''}</pre>
        </div>
        """))

    @staticmethod
    def agent_response(text):
        display(Markdown(text))

    @staticmethod
    def separator():
        display(HTML('<hr style="border:none; border-top:2px solid #dadce0; margin:16px 0;">'))

    @staticmethod
    def stopped():
        display(HTML("""
        <div style="background:#fce8e6; border:1px solid #c5221f; border-radius:8px; padding:12px 16px; margin:8px 0; text-align:center;">
            <b style="color:#c5221f;">Session ended.</b>
        </div>
        """))


async def chat_rich():
    """Multi-turn DS Agent with rich HTML UI."""
    messages = []
    ui = AgentUI()
    ui.header()

    while True:
        try:
            user_input = input('\nYou: ').strip()
        except (KeyboardInterrupt, EOFError):
            ui.stopped()
            break

        if not user_input:
            continue
        if user_input.lower() in ('quit', 'stop', 'exit'):
            ui.stopped()
            break
        if user_input.lower() == 'reset':
            messages = []
            ui.separator()
            ui.status('Conversation reset. Paste a new clinical note.')
            continue

        display(HTML(f"""
        <div style="background:#f0f0f0; border-radius:8px; padding:10px 16px; margin:8px 0; font-size:14px;">
            <b>👤 You:</b> {html_module.escape(user_input[:500])}{'...' if len(user_input) > 500 else ''}
        </div>
        """))

        messages.append({'role': 'user', 'content': user_input})
        ui.status('Agent is thinking...')

        final_content = ""
        try:
            async for chunk in agent.astream(
                {'messages': messages},
                stream_mode='updates'
            ):
                for node_name, node_output in chunk.items():
                    if node_name == 'agent':
                        msgs = node_output.get('messages', [])
                        for msg in msgs:
                            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                                for tc in msg.tool_calls:
                                    ui.tool_call(tc['name'], tc.get('args', {}))
                            if hasattr(msg, 'content') and msg.content:
                                if isinstance(msg.content, str) and msg.content:
                                    final_content = msg.content
                                elif isinstance(msg.content, list):
                                    for block in msg.content:
                                        if isinstance(block, dict) and block.get('type') == 'text':
                                            final_content += block['text']

                    elif node_name == 'tools':
                        msgs = node_output.get('messages', [])
                        for msg in msgs:
                            if hasattr(msg, 'content'):
                                ui.tool_result(
                                    getattr(msg, 'name', 'tool'),
                                    msg.content
                                )

        except KeyboardInterrupt:
            ui.status('Generation interrupted.')

        if final_content:
            ui.separator()
            ui.agent_response(final_content)
            messages.append({'role': 'assistant', 'content': final_content})

        ui.separator()

await chat_rich()
